# 1 · Linear Regression with Time Series

There are two kinds of features specific to time series:

| Feature | Built from | Models | Example |
|---|---|---|---|
| **Time-step** | the time index | *time dependence*: the value depends on *when* it is | time dummy `0, 1, 2, …` |
| **Lag** | past values of the target | *serial dependence*: the value depends on *what came before* | `y.shift(1)` |

Most of time-series ML comes down to engineering these two kinds of features. Linear regression then learns a weighted sum of them.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rc("figure", autolayout=True, figsize=(11, 4))
plot_params = dict(color="0.75", style=".-", markeredgecolor="0.25", markerfacecolor="0.25", legend=False)

DATA = Path("../input/store-sales-time-series-forecasting")   # change to where you unzipped the data

In [ ]:
store_sales = pd.read_csv(
    DATA / "train.csv",
    usecols=["store_nbr", "family", "date", "sales", "onpromotion"],
    dtype={"store_nbr": "category", "family": "category", "sales": "float32", "onpromotion": "uint32"},
    parse_dates=["date"],
)
store_sales["date"] = store_sales.date.dt.to_period("D")
store_sales = store_sales.set_index(["store_nbr", "family", "date"]).sort_index()

average_sales = store_sales.groupby("date").mean()["sales"]
average_sales.head()

## Time-step feature: the time dummy
`sales = w · time + b`: *w* is the average change per day.

In [ ]:
from sklearn.linear_model import LinearRegression

df = average_sales.to_frame()
df["time"] = np.arange(len(df.index))

X = df[["time"]]
y = df["sales"]

model = LinearRegression().fit(X, y)
y_pred = pd.Series(model.predict(X), index=X.index)

ax = y.plot(**plot_params, alpha=0.5)
y_pred.plot(ax=ax, linewidth=3, title="Time plot: sales vs. time dummy");

## Lag feature
`sales = w · lag_1 + b`: a strong relationship in the **lag plot** (y against its own past) means a lag feature will help.
Shifting creates NaNs at the start. Drop them and align the target.

In [ ]:
df["lag_1"] = df["sales"].shift(1)

X = df[["lag_1"]].dropna()
y, X = df["sales"].align(X, join="inner")

model = LinearRegression().fit(X, y)
y_pred = pd.Series(model.predict(X), index=X.index)

fig, ax = plt.subplots()
ax.plot(X["lag_1"], y, ".", color="0.25")
ax.plot(X["lag_1"], y_pred)
ax.set(aspect="equal", xlabel="lag_1", ylabel="sales", title="Lag plot");

In [ ]:
ax = y.plot(**plot_params)
y_pred.plot(ax=ax, title="Lag-1 model follows the recent past");

**Takeaways**
- The time dummy fits a trend line. The lag feature makes the forecast follow the most recent value.
- The best models usually combine both, plus the features in the next lessons.